# Metop (EPS) / AVHRR/3 (Advanced Very High Resolution Radiometer)

Example notebook from the [Satellite Data Catalog](https://alessandrosebastianelli.github.io/satcat/satellite.html?folder=metop/metop-avhrr). Each section below runs the Python example of one processing level.

Run the cells top to bottom. Some sources need a free account; the cell that needs credentials asks for them when it runs.

## L1B

Level 1B: calibrated, geolocated radiances/reflectances/brightness temperatures per orbit (EPS native format), in 1.1 km (full resolution) and 4 km (GAC) products.

Downloads one AVHRR L1B granule from the EUMETSAT Data Store (free account and consumer key/secret) and plots channel 4 with satpy.

In [ ]:
%pip install -q eumdac satpy netCDF4 h5netcdf matplotlib defusedxml

In [ ]:
import datetime as dt
import getpass
import shutil
import zipfile
from pathlib import Path

import eumdac

# Free consumer key / secret: https://api.eumetsat.int/api-key (EUMETSAT Data Store account)
token = eumdac.AccessToken((input("EUMETSAT consumer key: "), getpass.getpass("EUMETSAT consumer secret: ")))
datastore = eumdac.DataStore(token)
collection = datastore.get_collection("EO:EUM:DAT:METOP:AVHRRL1")

products = list(collection.search(dtstart=dt.datetime(2024, 6, 1, 12, 0), dtend=dt.datetime(2024, 6, 1, 12, 30)))
print(len(products), "AVHRR L1B products found")
if not products:
    raise SystemExit("Nothing in this time window - try another one")
product = products[0]
print(product)

out = Path("data")
out.mkdir(exist_ok=True)
# tens of MB per 3-minute granule
with product.open() as src, open(out / src.name, "wb") as dst:
    shutil.copyfileobj(src, dst)
for z in out.glob("*.zip"):                     # some collections arrive zipped
    with zipfile.ZipFile(z) as zf:
        zf.extractall(out / z.stem)

from glob import glob

import matplotlib.pyplot as plt
from satpy import Scene

files = sorted(glob("data/**/*.nat", recursive=True))
print(len(files), "files")
scn = Scene(filenames=files, reader="avhrr_l1b_eps")
print(scn.available_dataset_names())
scn.load(["4"])
scn["4"].plot(cmap="gray_r")                   # channel 4 brightness temperature
plt.title("AVHRR channel 4 (10.8 um)")
plt.show()